# 16S rPCA Analyses of the Longitudinal Acne Study

Date created: 10/18/2024

Notebook author: Britta De Pessemier

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

This notebook plots the following:
- rPCA beta diversity analysis for 16S data

In [32]:
# import Python packages
import pandas as pd
import numpy as np
from numpy import array
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
import scipy
import scipy.stats as ss
from skbio.stats.distance import permanova
import biom
from biom import load_table
from gemelli.rpca import rpca # (previously, auto_rpca)
from gemelli.utils import qc_rarefaction
from skbio.stats.distance import permanova, DistanceMatrix
from itertools import combinations
import qiime2 as q2
from qiime2 import Artifact

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="You passed a edgecolor/edgecolors .*"
)

In [33]:
# Function to draw an ellipse based on the covariance of the points
def draw_ellipse(mean, cov, ax, color, scale_factor=1.5):
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    width, height = 2 * np.sqrt(eigenvalues) * scale_factor

    # Add ellipse to the plot
    ell = Ellipse(
        xy=mean,
        width=width,
        height=height,
        angle=angle,
        edgecolor=color,
        facecolor=color,
        alpha=0.2,
        linewidth=0.5
    )
    ax.add_patch(ell)

In [ ]:
# Load Greengenes2 ASV taxonomy
# gg_taxonomy = pd.read_csv('../gg_taxonomy/2024.09.taxonomy.asv.tsv', sep='\t')

# gg_taxonomy

,Feature ID,Taxon,Confidence
0,GB-GCA-003568775.1-MWMI01000008.1,d__Bacteria; p__; c__; o__; f__; g__; s__,0.3
1,GB-GCA-001552015.1-CP010514.1,d__Bacteria; p__; c__; o__; f__; g__; s__,0.3
2,GB-GCA-000008085.1-AE017199.1,d__Bacteria; p__; c__; o__; f__; g__; s__,0.3
3,TAGCCGCACCCCAAGTGGTAGTCATTATTATTGGGCTTAAAGTGTT...,d__Bacteria; p__; c__; o__; f__; g__; s__,0.3
4,TACCCGCGCCACGAGTGGTGATCGCGATTATTGGGCCTAAAGGGTT...,d__Bacteria; p__; c__; o__; f__; g__; s__,0.3
...,...,...,...
23450263,TACCCGCGCTGCGAGTGGTCACCACGATTATTGGGCTTAGAGCGTT...,d__Archaea; p__; c__; o__; f__; g__; s__,0.3
23450264,TACCCGCGCCACGAGTGGTGATCGCGATTATTGGGCCTAAAGGGTT...,d__Archaea; p__; c__; o__; f__; g__; s__,0.3
23450265,CACTGGCAGTTCGGGTGGCAGTCGGTTCTATTGAGCCTAAAGCGTC...,d__Archaea; p__; c__; o__; f__; g__; s__,0.3
23450266,TACCCGCGCTGCAAGTGGTCACCACGATTATTGGGCTTAGAGCGTT...,d__Archaea; p__; c__; o__; f__; g__; s__,0.3


In [ ]:
# Set the random seed for reproducibility
np.random.seed(123)

# Define the color palette for the groups
group_colors = {
    "Healthy": "#3333B3",  # Color for Healthy
    "Acne_L": "#f16c52",   # Color for Acne Lesional
    "Acne_NL": "#5cbccb"   # Color for Acne Non-lesional
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/Tables/16S_V1-V3_Uncollapsed_absolute.biom'),
    ('174950', 'V1-V3', '../Data/16S/Tables/16S_V4_Uncollapsed_absolute.biom')
]

# Loop over both tables
for dataset_id, plot_title, biom_file in datasets:
    # print(f"Processing dataset: {dataset_id} - {plot_title}")
    
    # Load the biom table and perform RPCA
    biom_tbl = biom.load_table(biom_file)
    
    # Perform RPCA with auto rank estimation
    np.seterr(divide='ignore')
    ordination, distance = rpca(biom_tbl)

    # Read the metadata file
    metadata_file = '../Metadata/metadata_final_18062024.tsv'
    metadata_df = pd.read_csv(metadata_file, sep='\t')

    # Extract and view sample ordinations from RPCA result
    spca_df = pd.DataFrame(ordination.samples)
    spca_df["Group"] = spca_df.index.map(metadata_df.set_index("SampleID")["group"])
    
    # calculate permanova F-statistic
    pnova_res = permanova(distance, spca_df, "Group")
    p_value = pnova_res['p-value']
    
    # Plotting
    fig, ax = plt.subplots(1, 1, figsize=(7, 10))
    
    # Create scatter plot with different colors based on the Group column
    for group_name, group in spca_df.groupby('Group'):
        sns.scatterplot(
            data=group,
            x="PC1",
            y="PC2",
            facecolor=group_colors[group_name],  # Group-specific fill color
            edgecolor='k',                      # Thin black border
            alpha=0.8,                          # Transparency for the fill
            linewidth=0.5,                      # Thickness of the black border
            ax=ax,
            label=group_name_mapping[group_name],
            s=100                               # Size of the dots
        )
    
        # Calculate mean and covariance matrix for the group
        points = group[["PC1", "PC2"]].values
        mean = np.mean(points, axis=0)
        cov = np.cov(points, rowvar=False)
    
        # Draw the ellipse for the group (with scale factor applied)
        draw_ellipse(mean, cov, ax, group_colors[group_name], scale_factor=2)
    
        # Add a subtle star (8-pointed) at the mean location for each group
        ax.scatter(mean[0], mean[1], color=group_colors[group_name], marker=(8, 2, 0), s=400, edgecolor='black', zorder=5, linewidth=1.5)
    
    # Add axis labels and include the explained variance percentages
    proportion_explained = ordination.proportion_explained
    pc1_explained_variance = proportion_explained[0] * 100
    pc2_explained_variance = proportion_explained[1] * 100
    
    # Add axis labels and include the explained variance percentages
    ax.set_xlabel(f'PC1 ({pc1_explained_variance:.2f}%)', fontsize=18)
    ax.set_ylabel(f'PC2 ({pc2_explained_variance:.2f}%)', fontsize=18)

    # Simplify the ticks to avoid overcrowding
    yticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2]
    ax.set_yticks(yticklabels)
    ax.set_yticklabels(yticklabels, fontsize=18)

    xticklabels = [-0.2, -0.1, 0.0, 0.1, 0.2]
    ax.set_xticks(xticklabels)
    ax.set_xticklabels(xticklabels, fontsize=18)

    # Set plot title dynamically
    if plot_title == 'V1-V3':
        new_plot_title = '16S rRNA (V1-V3) rPCA Beta Diversity'
        plt.title(new_plot_title, fontsize=20)
    elif plot_title == 'V4':
        new_plot_title = '16S rRNA (V4) rPCA Beta Diversity'
        plt.title(new_plot_title, fontsize=20)
    
    # Set the thickness of the frame (axes spines)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
        
    # Customize legend
    ax.legend(
        frameon=False,
        fontsize=18,
        # title="Group",
        title_fontsize=18
    )

    # Adjust plot style and save
    ax.spines["right"].set_visible(True)
    ax.spines["top"].set_visible(True)

    # Extract feature loadings
    feature_loadings = ordination.features  # Loadings for the features
    features_df = pd.DataFrame(feature_loadings, columns=['PC1', 'PC2'], index=ordination.features.index)
    
    # Print the top 5 ASVs driving separation
    top_5_asvs = top_features_df.index.tolist()
    print("Top 5 ASVs driving separation:")
    for i, asv in enumerate(top_5_asvs, 1):
        print(f"{i}. {asv}")


    # # Calculate the contribution of each feature using the Euclidean norm
    # features_df['contribution'] = (features_df['PC1']**2 + features_df['PC2']**2)**0.5

    # # Select the top 5 features with the largest contributions
    # top_features_df = features_df.nlargest(5, 'contribution')

    # # Scale the feature loadings for visualization
    # scale_factor = 0.3  # Adjust to control arrow length relative to the plot
    # top_features_df['PC1_scaled'] = top_features_df['PC1'] * scale_factor
    # top_features_df['PC2_scaled'] = top_features_df['PC2'] * scale_factor

    # # Draw arrows and annotate the top 5 features driving separation
    # for feature_name, row in top_features_df.iterrows():
    #     ax.arrow(
    #         0, 0,                       # Start arrow at origin
    #         row['PC1_scaled'], row['PC2_scaled'], 
    #         color='black',              # Arrow color
    #         width=0.001,                # Arrow thickness
    #         head_width=0.005,            # Arrowhead width
    #         length_includes_head=True,  # Include arrowhead in total length
    #         zorder=4
    #     )
    #     # Add text label for the feature near the arrow tip
    #     ax.text(
    #         row['PC1_scaled'] * 1.1, row['PC2_scaled'] * 1.1,  # Slightly offset from the arrow tip
    #         feature_name, 
    #         fontsize=10, 
    #         color='black'
    #     )

    plt.tight_layout()
    
    plt.savefig(f"../Figures/16S_Figures/rPCA/{plot_title}_rPCA.png", dpi=600)


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11485/3946142812.py:78: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pc1_explained_variance = proportion_explained[0] * 100
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11485/3946142812.py:79: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pc2_explained_variance = proportion_explained[1] * 100
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11485/3946142812.py:161: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Top 5 ASVs driving separation:
1. TACGTAGGGTGCAAGCGTTAATCGGAATTATTGGGCGTAAAGCGAGTGCAGACGGTTACTTAAGCCAGATGTGAAATCCCCAAGCTTAACTTGGGACGTGCATTTGGAACTGGGTGACTAGAGTGTGTCAGAGGGAGGTAGAATTCCACA
2. TACGTAGGGTGCGAGCGTTAATCGGAATTATTGGGCGTAAAGCGAGTGCAGACGGTTGTTTAAGCCAGATGTGAAATCCCCGAGCTTAACTTGGGACGTGCATTTGGAACTGGATAACTAGAGTGTGTCAGAGGGAGGTAGAATTCCACA
3. TACGGAAGGTCCAGGCGTTATCCGGATTTATTGGGTTTAAAGGGAGCGTAGGCGGATTATTAAGTCAGTGGTGAAAGACGGTGGCTCAACCATCGTTAGCCATTGAAACTGGTAGTCTTGAGTGCAGACAGGGATGCTGGAACTCGTGGT
4. TACGTAGGGTGCGAGCGTTAATCGGAATTACTGGGCGTAAAGCGGGCGCAGACGGTTACTTAAGCAGGATGTGAAATCCCCGGGCTCAACCTGGGAACTGCGTTCTGAACTGGGTGACTAGAGTGTGTCAGAGGGAGGTAGAATTCCACG
5. TACGTAGGTGGCAAGCGTTGTCCGGATTTATTGGGCGTAAAGCGAGCGCAGGCGGAAGAATAAGTCTGATGTGAAAGCCCTCGGCTTAACCGAGGAACTGCATCGGAAACTGTTTTTCTTGAGTGCAGAAGAGGAGAGTGGAACTCCATG


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11485/3946142812.py:78: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pc1_explained_variance = proportion_explained[0] * 100
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11485/3946142812.py:79: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  pc2_explained_variance = proportion_explained[1] * 100
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_11485/3946142812.py:161: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Top 5 ASVs driving separation:
1. ATTGAACGCTGGCGGCATGCTTTACACATGCAAGTCGAACGGCAGCGAGGAGAAGCTTGCTTCTCTGTCGGCGAGTGGCGAACGGGTGAGTATAGCATCGGAACGTGCCAAGTAGTGGGGGATAACCAAACGAAAGTTTGGCTAATACCG
2. GACGAACGCTGGCGGCGTGCCTAATACATGCAAGTAGAACGCTGAAGGAGGAGCTTGCTTCTCTGGATGAGTTGCGAACGGGTGAGTAACGCGTAGGTAACCTGCCTGGTAGCGGGGGATAACTATTGGAAACGATAGCTAATACCGCAT
3. GATGAACGCTGGCGGCGTGCCTAATACATGCAAGTCGAGCGAACAGACGAGGAGCTTGCTCCTCTGACGTTAGCGGCGGACGGGTGAGTAACACGTGGATAACCTACCTATAAGACTGGGATAACTTCGGGAAACCGGAGCTAATACCGG
4. GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCTCCAGCTTGCTGGAGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTAATCTGCCCTGCACTCTGGGATAAGCCTGGGAAACTGGGTCTAATACTGGAT
5. GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGAAGAGCGATGGAAGCTTGCTTCTATCAATCTTAGTGGCGAACGGGTGAGTAACGCGTAATCAACCTGCCCTTCAGAGGGGGACAACAGTTGGAAACGACTGCTAATACC
